# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 4.5 - Deployment

In this notebook, we are going to deploy our fine-tuned Qwen 2.5 - 7B Instruct model to a SageMaker real-time inference endpoint.

---

## Prerequisites
### AWS Access Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

project_prefix = "humanlike-rlaif"
base_model_jumpstart_id = "huggingface-llm-qwen2-5-7b-instruct"
base_model_shortname = "qwen25-7b"

### Define helper functions

In [ ]:
import time

def wait_for_endpoint(sm_client, endpoint_name):
    while True:
        status = sm_client.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        print(f"Endpoint status: {status}")
        if status in ["InService", "Failed"]:
            return status
        time.sleep(30)

def wait_for_inference_component(sm_client, component_name):
    while True:
        status = sm_client.describe_inference_component(
            InferenceComponentName=component_name
        )["InferenceComponentStatus"]
        print(f"Inference component status: {status}")
        if status in ["InService", "Failed"]:
            return status
        time.sleep(30)

## Prepare and create inference endpoint
### Define endpoint parameters

In [ ]:
model_package_group_name = f"{project_prefix}-{base_model_shortname}"
model_package_version = "1"

endpoint_config_name = f"{model_package_group_name}-endpoint-config"
endpoint_name = f"{model_package_group_name}-endpoint"
ic_name = f"{model_package_group_name}-ic"

instance_type = "ml.g5.2xlarge"
instance_count = 1
health_check_timeout = 700

### Get fine-tuned model S3 artifact path

In [ ]:
import os
from sagemaker.core.resources import ModelPackage, ModelPackageGroup

model_package_group = ModelPackageGroup.get(model_package_group_name)
model_package_arn = f"{model_package_group.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
model_package = ModelPackage.get(model_package_arn)

model_s3_uri = os.path.join(
    model_package.inference_specification.containers[0].model_data_source.s3_data_source.s3_uri,
    "checkpoints", "hf_merged"
)
if not model_s3_uri.endswith("/"):
    model_s3_uri += "/"
print(f"Fine-tuned model S3 path: {model_s3_uri}")

### Create Endpoint Config

In [ ]:
from rich.pretty import pprint

endpoint_config_response = sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "InstanceType": instance_type,
            "InitialInstanceCount": instance_count,
            "ModelDataDownloadTimeoutInSeconds": health_check_timeout,
            "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
        }
    ],
)
pprint(endpoint_config_response)

### Create the Endpoint and wait until it is ready

In [ ]:
endpoint_response = sm_client.create_endpoint(
    EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name
)
pprint(endpoint_response)

wait_for_endpoint(sm_client, endpoint_name)

### Create the Model from the ModelPackage

In [ ]:
import json
from sagemaker.core import image_uris

In [ ]:
image_uri = image_uris.retrieve(
    framework="djl-lmi",
    region=sess.boto_region_name,
    version="latest",
)

image_uri = image_uri.split("/")[0] + "/djl-inference:0.36.0-lmi18.0.0-cu128"

image_uri

In [ ]:
env = {
    "HF_MODEL_ID": "/opt/ml/model",  # path to where sagemaker stores the model
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_MODEL_LOADING_TIMEOUT": "3600",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "SERVING_FAIL_FAST": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_DTYPE": "bf16",
    "OPTION_QUANTIZE": "fp8",
    "OPTION_MAX_MODEL_LEN": json.dumps(1024 * 32)
}

In [ ]:
model_response = sm_client.create_model(
    ModelName= f"{project_prefix}-{base_model_shortname}",
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": image_uri,
        "Environment": env,
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": model_s3_uri,
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
    },
)

pprint(model_response)

### Create the Inference Component

In [ ]:
ic_response = sm_client.create_inference_component(
    InferenceComponentName=ic_name,
    EndpointName=endpoint_name,
    VariantName="AllTraffic",
    Specification={
        "ModelName": f"{project_prefix}-{base_model_shortname}",
        "ComputeResourceRequirements": {
            "MinMemoryRequiredInMb": 12288,
            "NumberOfAcceleratorDevicesRequired": 1,
        },
    },
    RuntimeConfig={"CopyCount": 1},
)

pprint(ic_response)

In [ ]:
wait_for_inference_component(sm_client, ic_name)

## Test the inference endpoint
### Iterator class for streaming inference
Utility class to parse streaming responses

In [ ]:
import io

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()

            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]

            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise

            if "PayloadPart" not in chunk:
                continue

            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

Utility function to parse model answer

In [ ]:
def parse_streaming_response(line_str):
    """Parse a single streaming response line and return content if found."""
    if not line_str.strip() or line_str.strip() == "data: [DONE]":
        return None

    if line_str.startswith("data: "):
        line_str = line_str[6:]

    try:
        data = json.loads(line_str)
        if "choices" in data:
            for choice in data["choices"]:
                if "delta" in choice and "content" in choice["delta"]:
                    return choice["delta"]["content"]
    except json.JSONDecodeError:
        pass

    return None

In [ ]:
prompt = """
What's your honest take on pineapple on pizza?
"""

In [ ]:
request_body = {
    "model_name": ic_name,
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ],
    "max_tokens": 4096,
    "temperature": 0.3,
    "top_p": 0.9,
    "stop": ["<|im_end|>"],
    "stream": True,
}

smr_client = boto3.client("sagemaker-runtime", region_name=sess.boto_region_name)

response = smr_client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(request_body),
    ContentType="application/json",
)

generated_text = ""

for line in LineIterator(response["Body"]):
    if line:
        content = parse_streaming_response(line.decode("utf-8"))
        if content:
            generated_text += content
            print(content, end="", flush=True)

## Clean up
### Delete the created resources

In [ ]:
# Delete inference component
sm_client.delete_inference_component(InferenceComponentName=ic_name)

# Delete model
sm_client.delete_model(ModelName=f"{project_prefix}-{base_model_shortname}")


# Delete endpoint config (optional)
sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)

# Delete endpoint (optional - if you want to remove the endpoint too)
sm_client.delete_endpoint(EndpointName=endpoint_name)